In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

####1. Create the transactions table

auto optimize is turned off so we can understand how optimization works without letting DBX take control

In [0]:
%sql
CREATE OR REPLACE TABLE transactions (
  txn_id      INT,
  customer_id INT,
  amount      DOUBLE,
  category    STRING,
  txn_date    DATE
)
USING DELTA
COMMENT 'Transactions table — used to demonstrate OPTIMIZE and VACUUM'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'false',
  'delta.autoOptimize.autoCompact' = 'false'
);

####2. Write ten rows as ten separate INSERTs

In [0]:
%sql
INSERT INTO transactions VALUES (1,  101, 49.99,  'Electronics', '2024-01-05');
INSERT INTO transactions VALUES (2,  102, 129.00, 'Furniture',   '2024-01-06');
INSERT INTO transactions VALUES (3,  103, 19.50,  'Electronics', '2024-01-07');
INSERT INTO transactions VALUES (4,  101, 89.99,  'Clothing',    '2024-01-08');
INSERT INTO transactions VALUES (5,  104, 249.00, 'Furniture',   '2024-01-09');
INSERT INTO transactions VALUES (6,  102, 34.99,  'Electronics', '2024-01-10');
INSERT INTO transactions VALUES (7,  105, 15.00,  'Clothing',    '2024-01-11');
INSERT INTO transactions VALUES (8,  103, 199.99, 'Electronics', '2024-01-12');
INSERT INTO transactions VALUES (9,  101, 74.50,  'Furniture',   '2024-01-13');
INSERT INTO transactions VALUES (10, 106, 9.99,   'Clothing',    '2024-01-14');

>Ten rows, ten separate INSERT statements. 
In a real pipeline this happens naturally — streaming micro-batches, incremental loads from an API, one row at a time event processing. 
The result is the same: one new Parquet file per write.

####3. DESCRIBE DETAIL: see the file count before OPTIMIZE

>Look at numFiles — ten files.\
sizeInBytes will be very small

In [0]:
%sql
DESCRIBE DETAIL transactions;

####4. Run OPTIMIZE

In [0]:
%sql
OPTIMIZE transactions;

####5 DESCRIBE DETAIL after OPTIMIZE: file count drops

>umFiles has dropped to one. \
The data is unchanged — ten rows are still there — but they are now in one well-sized file instead of ten tiny ones. \
A query that scans this table now opens one file instead of ten. At scale, that difference compounds dramatically.

In [0]:
%sql
DESCRIBE DETAIL transactions;

In [0]:
%sql
describe history transactions

####6. Z-ORDER mention

>OPTIMIZE also accepts a ZORDER BY clause — you can see the syntax here. \
Z-Ordering physically co-locates rows with similar column values in the same files, which helps queries that filter on that column skip irrelevant files entirely. We will cover this properly in the Liquid Clustering.

In [0]:
%sql
-- OPTIMIZE transactions ZORDER BY (category);

####7.  VACUUM: dry run, safety check

In [0]:
%sql
VACUUM transactions DRY RUN;

####8. Attempt VACUUM without overrides — expect the safety error

In [0]:
%sql
VACUUM transactions RETAIN 0 HOURS;

>Delta's safety check fires. 
The error tells you that a retention of less than 7 days risks making concurrent readers see inconsistent data, and it refuses to proceed. This is a deliberate protection — it stops you from accidentally running VACUUM so aggressively that you break time travel or corrupt in-flight reads.

>On Databricks since December 2025, the RETAIN 0 HOURS clause in this command is actually ignored. Even if the safety check were disabled, VACUUM would not use that value. Retention is now controlled exclusively through a table property, not through the command syntax. We will set that property in a moment. The syntax VACUUM table RETAIN X HOURS still runs without a syntax error — it just silently uses the table property value instead of what you wrote. 

####10. Set retention to zero via table property

In [0]:
%sql
ALTER TABLE transactions
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

>Setting it to interval 0 hours tells VACUUM to remove all logically-removed files immediately,\
regardless of when they were marked as removed. In production you would set this to something like interval 30 days if you need a month of time travel history,

####11. Run VACUUM for real

In [0]:
%sql
VACUUM transactions;


>VACUUM has run. \
The old pre-OPTIMIZE files have been physically deleted from S3. The table still has ten rows in one compacted file — the current state is unchanged. But the old files are gone.

####12. Attempt time travel to older version — expect it to fail

In [0]:
%sql
desc history transactions


In [0]:
%sql
SELECT * FROM transactions

In [0]:
%sql
SELECT * FROM transactions VERSION AS OF 5